# Healome Aging Clock — Evaluation & Benchmarking

This notebook demonstrates how to evaluate a biological age model using
the Healome benchmarking framework. Two tracks:

1. **Age Prediction**: MAE, RMSE, R², Pearson r, subgroup analysis
2. **Survival Analysis**: Kaplan-Meier curves, Cox PH, concordance index

Use this to benchmark your own model against the Healome baselines.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from healome_clock import HealomeClock
from healome_clock.models.tree import STANDARD_21_FEATURES, EXTENDED_35_FEATURES
from healome_clock.evaluation.metrics import (
    compute_age_metrics, compute_age_bucket_metrics,
    compute_subgroup_metrics, print_metrics,
)
from healome_clock.evaluation.survival import (
    prepare_survival_data, compute_kaplan_meier, compute_cox_hazard_ratios,
)
from healome_clock.evaluation.leaderboard import (
    create_submission, save_submission, compare_submissions,
)

## Step 1: Load your test data

You need a DataFrame with NHANES biomarker columns + RIDAGEYR (chronological age).

In [ ]:
# Load your NHANES test data (update path)
# dd_age = pd.read_csv("your_test_data.csv")

# For demonstration, we show the workflow with synthetic data
np.random.seed(42)
n = 1000
y_true = np.random.uniform(20, 80, n)
y_pred = y_true + np.random.normal(0, 6, n)  # MAE ~5 years

print(f"Test samples: {n}")

## Track 1: Age Prediction Metrics

In [ ]:
metrics = compute_age_metrics(y_true, y_pred)
print_metrics(metrics, "Your Model")

In [ ]:
# Subgroup analysis by age bucket
bucket_metrics = compute_age_bucket_metrics(y_true, y_pred)
print(bucket_metrics.to_string(index=False))

## Track 2: Survival Analysis

Requires mortality data merged with biomarker data.
See `training.ipynb` for how to load mortality data.

In [ ]:
# Example workflow (requires real data):
#
# from healome_clock.data.mortality import load_mortality_data, merge_with_mortality
#
# mortality = load_mortality_data(download=True)
# merged = merge_with_mortality(dd_age, mortality)
# merged["bio_age"] = your_model.predict(merged[features])
#
# surv = prepare_survival_data(merged)
# km = compute_kaplan_meier(surv)
# cox = compute_cox_hazard_ratios(surv)
# print(f"Concordance: {cox['concordance']:.4f}")

print("Survival analysis requires NHANES data + mortality files.")
print("See training.ipynb for the full pipeline.")

## Create a Leaderboard Submission

In [ ]:
submission = create_submission(
    model_name="My Aging Clock",
    y_true=y_true,
    y_pred=y_pred,
    authors="Your Name",
    description="GradientBoosting with custom features",
    model_type="GradientBoostingRegressor",
    n_features=21,
    training_data="NHANES 2003-2020",
    test_split_method="random 70/30, seed=3454",
    concordance=0.83,  # from Cox PH if available
)

import json
print(json.dumps(submission, indent=2))

In [ ]:
# Save submission
save_submission(submission, "../benchmarks/submissions/my_model.json")

# Compare all submissions
leaderboard = compare_submissions("../benchmarks/baselines/")
print("\nLeaderboard:")
print(leaderboard.to_string(index=False))

---

**To submit to the leaderboard:**
1. Run your model on the NHANES test set (same split: test_size=0.3, random_state=3454)
2. Generate your submission JSON using `create_submission()`
3. Open a PR adding your JSON to `benchmarks/submissions/`

See [benchmarks/README.md](../benchmarks/README.md) for full instructions.